In [1]:
# packages and scripts
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import re
import os
import scripts.helper_functions as func

# load data
BASEDIR = os.path.expanduser('~/code/ast_mouse/')
tmp_data = pd.read_csv(BASEDIR + 'data/raw/all_final.csv')

# change column names so they are consistent 
tmp_data.rename(columns={'block': 'Block', 'correct': 'Correct?', 'side_rwd': 'RewardedLocation',
                                         'stim_rwd': 'RewardedStim', 'left_stim': 'LeftStim', 'right_stim': 'RightStim',
                                        'session': 'Session', 'mouse': 'Animal'}, inplace=True)

# make columns consistent across datasets
datasets = [tmp_data]
for i in range(len(datasets)):
    datasets[i] = func.consistent(datasets[i])

all_data = datasets[0]

# list of mice that may need to be removed
bad_mice = ['AMA1-00703']
retest_mice = ['AMA1-00613-retest','AMA1-00615-retest','AMA1-00616-retest']

# remove just the bad ones for now
#all_data = all_data[~all_data['animal'].isin(bad_mice)]

# some basic preprocessing, removes blocks we don't want
all_data = all_data[all_data['block'] != '1_EDSR']
all_data = all_data[all_data['block'] != 'EDSR']
all_data = all_data[all_data['block'] != 'NoRule']

# fix correct column and make sure vals are ints
all_data['correct?'] = all_data['correct?'].replace('x','0')
all_data['correct?'] = all_data['correct?'].fillna('0')
all_data['correct?'] = all_data['correct?'].astype(int)

print(all_data.groupby('condition')['animal'].nunique())

condition
GH_M_Control     8
GH_M_Dreadds     8
GH_M_mCherry     7
ISO_M_Control    7
ISO_M_Dreadds    6
ISO_M_mCherry    6
Name: animal, dtype: int64


In [4]:
# remove alfredos mice
data = all_data

# get a list of mice and blocks, we'll loop over these later
mice = data.animal.unique()
blocks = data.block.unique()
blocks = [b for b in blocks if b != 'NoRule']

# remove na from correct? col
data.dropna(subset = ['correct?'], inplace = True)

# correct some stimuli names so they are consistent for the model
# please don't change order as regex searches and replaces substrings
# e.g car is substring of cardboard, so creates carddboard, which we the change to just card
data = data.replace('Metal Strip', 'MetalStrip', regex = True)
data = data.replace('Metal strip ', 'MetalStrip', regex = True)
data = data.replace('Metal strip', 'MetalStrip', regex = True)
data = data.replace('Poppuri', 'Popurri', regex = True)
data = data.replace('Car', 'Card', regex = True)
data = data.replace('Carddboard', 'Card', regex = True)
data = data.replace('WhBedd', 'WhBed', regex = True)
data = data.replace('Whbed', 'WhBed', regex = True)
data = data.replace('Turmeric', 'Tumeric', regex = True)
data = data.replace('Tumeric ', 'Tumeric', regex = True)
data = data.replace('Cardd', 'Card', regex = True)
data = data.replace('Cloves', 'Clove', regex = True)
data = data.replace('left', 'Left', regex = True)
data = data.replace('tissue', 'Tissue', regex = True)


# now run the function to clean the data
clean_data = func.clean_ied(data)
# now lets standardize the features
clean_data = func.standardize_features(clean_data)

# if you want to remove the 2nd day from each block
clean_data = clean_data[clean_data['day'] == '1']

# remap condition
# Example mapping dictionary
mapping = {
    'GH_M_Control': 'GH',
    'GH_M_mCherry': 'GH',
    'ISO_M_Control': 'ISO',
    'ISO_M_mCherry': 'ISO',
    'GH_M_Dreadds': 'GH DREADDs',
    'ISO_M_Dreadds': 'ISO DREADDs'
}

clean_data['condition'] = clean_data['condition'].replace(mapping)

# now lets save the clean data tabular data in csv format 
clean_data.to_csv(BASEDIR + 'data/clean/clean_data.csv', index = False)


In [6]:
# now let's use the filter_block function to remove trials and edit blocks where their might be errors 
filtered_data = func.filter_blocks(clean_data)
clean_filtered_data = filtered_data[0]
errors_df = filtered_data[1]

# save
clean_filtered_data.to_csv(BASEDIR + 'data/clean/clean_filtered_data.csv', index = False)

In [8]:
## now run through function to build the datalist
data = clean_filtered_data

# remove_retest mice
data = data[~data['mouse'].isin(retest_mice)]

## split by group
mice_ghd = data.loc[data['condition'] == 'GH DREADDs']['mouse'].unique()
ghc = ['GH', 'GH mCherry']
mice_ghc = data.loc[data['condition'].isin(ghc)]['mouse'].unique()
mice_isod = data.loc[data['condition'] == 'ISO DREADDs']['mouse'].unique()
isoc = ['ISO', 'ISO mCherry']
mice_isoc = data.loc[data['condition'].isin(isoc)]['mouse'].unique()

# all mice
mice = data['mouse'].unique()

## run function to wrangle the data into a format the RL models will use 
clean_model_data_ghd = [func.make_ast_model_data(data, mouse) for mouse in mice_ghd]
clean_model_data_isod = [func.make_ast_model_data(data, mouse) for mouse in mice_isod]
clean_model_data_ghc = [func.make_ast_model_data(data, mouse) for mouse in mice_ghc]
clean_model_data_isoc = [func.make_ast_model_data(data, mouse) for mouse in mice_isoc]

# all mice
clean_model_data = [func.make_ast_model_data(data, mouse) for mouse in mice]

## save clean datasets as pickle files
LISTDIR = BASEDIR + 'data/datalists/em'
os.makedirs(LISTDIR, exist_ok=True)
with open(LISTDIR + '/ghd_r0_!2.pkl', 'wb') as file:
    pickle.dump(clean_model_data_ghd, file)
with open(LISTDIR + '/isod_r0_!2.pkl', 'wb') as file:
    pickle.dump(clean_model_data_isod, file)
with open(LISTDIR + '//ghc_r0_!2.pkl', 'wb') as file:
    pickle.dump(clean_model_data_ghc, file)
with open(LISTDIR + '/isoc_r0_!2.pkl', 'wb') as file:
    pickle.dump(clean_model_data_isoc, file)

# save all
with open(LISTDIR + '/all_r0_!2.pkl', 'wb') as file:
    pickle.dump(clean_model_data, file)

In [9]:
# let's check the data 
func.sanity_check(clean_model_data)

Passed: Last 8 choices are all rewarded!
Passed: Inital dimension coded correctly!
